Import secrets and make sure they are valid

In [0]:
key = dbutils.secrets.get("motorsport", "kafka_api_key")
secret = dbutils.secrets.get("motorsport", "kafka_api_secret")
print(len(key), repr(key[:2]), repr(key[-2:]))
print(len(secret), '"' in secret, '\\' in secret, '\n' in secret, secret != secret.strip())

Check if kafka is reachable

In [0]:
bootstrap = dbutils.secrets.get("motorsport", "kafka_bootstrap")
key = dbutils.secrets.get("motorsport", "kafka_api_key").strip()
secret = dbutils.secrets.get("motorsport", "kafka_api_secret").strip()

jaas = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
    f'required username="{key}" password="{secret}";'
)

df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap)
    .option("subscribe", "motorsport.public.vehicles")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print(df.count())
display(df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "topic", "partition", "offset").limit(5))

create schemas

In [0]:
%sql
DROP CATALOG IF EXISTS motorsport CASCADE;
CREATE CATALOG motorsport
  MANAGED LOCATION 's3://motorsport-data-lake/uc/';
CREATE SCHEMA IF NOT EXISTS motorsport.bronze;
CREATE SCHEMA IF NOT EXISTS motorsport.silver;
CREATE SCHEMA IF NOT EXISTS motorsport.gold;

Check if S3 is reachable

In [0]:
spark.read.parquet("s3://motorsport-data-lake/landing/").limit(5).show()

In [0]:
checkpoint_path = "s3://motorsport-data-lake/checkpoints"
dbutils.fs.rm(checkpoint_path, recurse=True)